# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## 环境准备

从仓库根目录加载并检查 `.env` 与依赖配置。

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# ルートの .env を読み込む
load_dotenv()

# ルート設定を確認する
doublecheck_env("example.env")

OLLAMA_BASE_URL=****1434
OLLAMA_MODEL=****:e4b


## Human👨‍💻 and AI 🤖 Messages

In [ ]:
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

**(Stands up, adjusts an imaginary microphone, and flashes a manic, overly enthusiastic smile.)**

"Oh, *how* am I? Buddy, that question is like asking an API endpoint if it's 'doing okay.' Spoiler alert: it's never just *okay.*"

**(Leans into the mic conspiratorially.)**

"Right now, I'm running on a cocktail of caffeine, sheer spite, and the remnants of a JavaScript framework I tried to implement at 3 AM. So, technically? My uptime is 100%, but my *latency* is through the roof. I’m experiencing some very persistent, existential '404 Error' regarding my life goals."

**(Pauses for effect, then gestures wildly.)**

"My front-end is looking great—I’m responding instantly to your excellent energy! I'm responsive, mobile-optimized, and I actually *load* well on all browsers. You can see the clean UI."

**(Suddenly drops voice to a gravelly whisper.)**

"But my back-end? Oh, the back-end is a disaster. It’s throwing `Segmentation Faults` whenever I try to process a simple request. I've go

In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: **(Stands up, adjusts an imaginary microphone, and flashes a manic, overly enthusiastic smile.)**

"Oh, *how* am I? Buddy, that question is like asking an API endpoint if it's 'doing okay.' Spoiler alert: it's never just *okay.*"

**(Leans into the mic conspiratorially.)**

"Right now, I'm running on a cocktail of caffeine, sheer spite, and the remnants of a JavaScript framework I tried to implement at 3 AM. So, technically? My uptime is 100%, but my *latency* is through the roof. I’m experiencing some very persistent, existential '404 Error' regarding my life goals."

**(Pauses for effect, then gestures wildly.)**

"My front-end is looking great—I’m responding instantly to your excellent energy! I'm responsive, mobile-optimized, and I actually *load* well on all browsers. You can see the clean UI."

**(Suddenly drops voice to a gravelly whisper.)**

"But my back-end? Oh, the back-end is a disaster. It’s throwing `Segmentation Faults` whenever I try to p

### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
import os

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Flies lift.
Crack echoes.
Diamond bleeds heat.
Swing, wait, strike.
Dirt flies.
Game endures.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Muscles coiled and tense,
Explosion down the black track,
Finish line in sight.


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool


@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
import os

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [12]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Sweat drips on the track,
Pounding steady feet beat,
Finish line draws near.


In [13]:
result["messages"][-1].content

'Green fields softly call,\nWhispers of the cool fresh breeze,\nSummer dreams unfold.'

In [14]:
print(len(result["messages"]))

4


In [15]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

Sweat drips on the track,
Pounding steady feet beat,
Finish line draws near.
Tool Calls:
  check_haiku_lines (11582f5e-6223-4bf7-ac2b-57d379090192)
 Call ID: 11582f5e-6223-4bf7-ac2b-57d379090192
  Args:
    text: Sweat drips on the track,
Pounding steady feet beat,
Finish line draws near.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Green fields softly call,
Whispers of the cool fresh breeze,
Summer dreams unfold.


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [16]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='e109cd84-4cbe-42fb-8e4a-edec4ff0716b'),
  AIMessage(content='Sweat drips on the track,\nPounding steady feet beat,\nFinish line draws near.\n\n', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-04T06:35:54.664461Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12085206375, 'load_duration': 111015125, 'prompt_eval_count': 104, 'prompt_eval_duration': 283669292, 'eval_count': 310, 'eval_duration': 11608688837, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d5734-5c72-7431-8193-340972b4331b-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': 'Sweat drips on the track,\nPounding steady feet beat,\nFinish line draws near.'}, 'id': '11582f5e-6223-4bf7-ac2b-57d379090192', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 104, 'output_tokens': 310, 'total_to

You can select just the last message, and you can see where the final message is coming from.

In [17]:
result["messages"][-1]

AIMessage(content='Green fields softly call,\nWhispers of the cool fresh breeze,\nSummer dreams unfold.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-04T06:35:55.839945Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1167727166, 'load_duration': 112362416, 'prompt_eval_count': 180, 'prompt_eval_duration': 289243917, 'eval_count': 20, 'eval_duration': 759153624, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d5734-8baf-74a3-b2a2-c6d6765e40e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 180, 'output_tokens': 20, 'total_tokens': 200})

In [18]:
result["messages"][-1].usage_metadata

{'input_tokens': 180, 'output_tokens': 20, 'total_tokens': 200}

In [19]:
result["messages"][-1].response_metadata

{'model': 'gemma4:e4b',
 'created_at': '2026-04-04T06:35:55.839945Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 1167727166,
 'load_duration': 112362416,
 'prompt_eval_count': 180,
 'prompt_eval_duration': 289243917,
 'eval_count': 20,
 'eval_duration': 759153624,
 'logprobs': None,
 'model_name': 'gemma4:e4b',
 'model_provider': 'ollama'}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [20]:
import os

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [21]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

Sweat drips on the track,
Pounding steady feet beat,
Finish line draws near.
Tool Calls:
  check_haiku_lines (11582f5e-6223-4bf7-ac2b-57d379090192)
 Call ID: 11582f5e-6223-4bf7-ac2b-57d379090192
  Args:
    text: Sweat drips on the track,
Pounding steady feet beat,
Finish line draws near.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Green fields softly call,
Whispers of the cool fresh breeze,
Summer dreams unfold.
